## modulo 3

### Laboratorio mod 3

In [1]:
import time


def retry(max_retries=3):
    def decorator(func):
        def wrapper():
            for i in range(max_retries):
                try:
                    return func()

                except Exception:
                    wait = 2**i
                    print(f"Reintentando en {wait}s")
                    time.sleep(wait)

        return wrapper

    return decorator


retry(max_retries=5)("hola")()

Reintentando en 1s
Reintentando en 2s
Reintentando en 4s
Reintentando en 8s
Reintentando en 16s


## modulo 4

In [ ]:
# ruff: noqa: F811
from dataclasses import dataclass

# DATACLASSES
"""
Evitan escribir mucho código repetitivo.
Automáticamente genera:

constructor,
repr,
equality,
etc."""


@dataclass
class User:
    name: str
    age: int

In [ ]:
# ruff: noqa: F811
# ruff: noqa: E402

# attrs
"""
librerías,
sistemas complejos,
validaciones avanzadas."""  # noqa: E402
import attrs  # noqa: E402


@attrs.define
class User:
    name: str
    age: int


# pydantic
"""
lanza errores,
valida tipos,
serializa JSON."""
from pydantic import BaseModel


class User(BaseModel):
    name: str
    age: int

#### Laboratorio mod 4

In [10]:
from dataclasses import dataclass


@dataclass
class Order:
    product: str
    quantity: int
    price: float

    @property
    def subtotal(self):
        return self.quantity * self.price

    @property
    def tax(self):
        return self.subtotal * 0.16

    @property
    def total(self):
        return self.subtotal + self.tax


o = Order(product="Laptop", quantity=2, price=15000)

print(o.subtotal)
print(o.tax)
print(o.total)

30000
4800.0
34800.0


### MODULO 5

In [ ]:
# Con tipado de datos mejor autocompletado, validación y documentación
def suma(a: int, b: int) -> int:
    return a + b


# pip install ruff
# ruff check .
"""
Detecta:

imports no usados,
variables no usadas,
errores de estilo,
malas prácticas,
bugs potenciales.
"""

# pip install black
# black .
"""Formateador de código automático."""

# pip install pre-commit
# pre-commit install
"""Ejecuta linters y formateadores antes de cada commit."""

#### LABORATORIO MODULO 5

In [16]:
from typing import Optional


def greet(name: Optional[str]) -> str:
    if name is None:
        return "Hola desconocido"

    return f"Hola {name}"


greet("Fernando")

'Hola Fernando'

### MODULO 6

## Laboratorio

In [20]:
import csv
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, filename="pipeline.log")

ruta_csv = Path("../data/ejemplo_mod6.csv")

ventas = []

logging.info("Leyendo CSV")

with open(ruta_csv, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        row["ventas"] = int(row["ventas"])

        ventas.append(row)

# Métricas
total = sum(v["ventas"] for v in ventas)

promedio = total / len(ventas)

resultado = {"total": total, "promedio": promedio, "registros": ventas}

logging.info("Exportando JSON")

with open("resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

logging.info("Pipeline finalizado")

### MODULO 7
#### LABORATORIO 

In [22]:
! pip install httpx

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
Using cached certifi-2026.4.22-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/6 [idna]
   ------ --------------------------------- 1/6 [h11]
   -------------------- ------------------- 3/6 [httpcore]
   -------------------- ------------------- 3/6 [httpcore]
   -------------------- ------------------- 3/6 [httpcore]
   -------------------------- ------------- 4/6 [anyio]
   -------------------------- ------------- 4/6 [anyio]
   -----------------

In [26]:
# Cliente con retries
import httpx


class APIClient:
    def __init__(self):
        self.client = httpx.Client(timeout=5)

    def get(self, url, retries=3):
        for intento in range(retries):
            try:
                response = self.client.get(url)

                response.raise_for_status()

                return response.json()

            except httpx.TimeoutException:
                wait = 2**intento

                print(f"Timeout. Retry en {wait}s")

                time.sleep(wait)

            except httpx.HTTPStatusError as e:
                print(f"HTTP error: {e}")

                break

            except httpx.RequestError as e:
                print(f"Request error: {e}")

                break

        return None


client = APIClient()

data = client.get("http://127.0.0.1:8000/api/v1/users")

print(data)

[{'id': '12fw342ej1', 'name': {'familyName': 'Muro', 'givenName': 'Rupert'}, 'age': 67}, {'id': '98ab12cd34', 'name': {'familyName': 'García', 'givenName': 'Lucía'}, 'age': 34}, {'id': '56gh78ij90', 'name': {'familyName': 'Pérez', 'givenName': 'Andrés'}, 'age': 45}, {'id': 'ab12cd34ef', 'name': {'familyName': 'López', 'givenName': 'María'}, 'age': 29}, {'id': 'cd34ef56gh', 'name': {'familyName': 'Ramírez', 'givenName': 'Jorge'}, 'age': 53}, {'id': '5bd75841-abae-4716-b670-865357d9a1d2', 'name': {'familyName': 'Muro', 'givenName': 'Rupert'}, 'age': 67}]


In [27]:
url = "https://example.com/archivo.zip"

with httpx.stream("GET", url) as response:
    with open("archivo.zip", "wb") as f:
        for chunk in response.iter_bytes():
            f.write(chunk)